In [90]:
!mkdir -p Support-Assistant/

In [91]:
!mkdir -p Support-Assistant/support_assistant


In [92]:
import os
from pathlib import Path


init_file_path = Path('Support-Assistant/support_assistant/__init__.py')
package_dir = init_file_path.parent

os.makedirs(package_dir, exist_ok=True)

if init_file_path.is_dir():
    os.rmdir(init_file_path)

with open(init_file_path, 'w') as f:
    f.write('"""Zepto Support Assistant package.\n\nModule 3:\n- Document ingestion\n- Local embeddings\n- ChromaDB retrieval\n- LangGraph orchestration\n- FastAPI API\n\n__version__ = "1.0.0"\n"""')


In [93]:
%%writefile Support-Assistant/support_assistant/config.py

import os
from pathlib import Path


BASE_DIR = Path('/content/Support-Assistant/support_assistant')

DOCS_DIR = BASE_DIR / "docs"

CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "zepto_support"


EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"


MOCK_LLM = os.getenv("MOCK_LLM", "1") == "1"

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

GROQ_MODEL = os.getenv(
    "GROQ_MODEL",
    "llama-3.1-8b-instant"
)


# Retrieval configuration

TOP_K = 1

SNIPPET_LENGTH = 200



# Mock confidence

MOCK_CONFIDENCE = 1.0

Writing Support-Assistant/support_assistant/config.py


In [94]:
%%writefile Support-Assistant/support_assistant/state.py

from typing import TypedDict, List, Optional


class SupportState(TypedDict, total=False):
    """
    Shared state passed between LangGraph nodes.
    """

    # Original user query
    query: str

    # Intent classification
    intent: str

    # Routing decision
    route: str

    # Retrieved context
    retrieved_context: str

    # IDs of retrieved chunks
    sources: List[str]

    # Final generated answer
    answer: str

    # Confidence score
    confidence: float

    # Optional error information
    error: Optional[str]

Writing Support-Assistant/support_assistant/state.py


In [95]:
%%writefile Support-Assistant/support_assistant/schemas.py

from typing import List

from pydantic import BaseModel, Field


class AskRequest(BaseModel):
    """
    Request schema for POST /ask
    """

    query: str = Field(
        ...,
        min_length=1,
        description="User's support question"
    )


class AskResponse(BaseModel):
    """
    Validated response returned by the support assistant.
    """

    answer: str = Field(
        ...,
        min_length=1
    )

    sources: List[str] = Field(
        default_factory=list
    )

    confidence: float = Field(
        ...,
        ge=0.0,
        le=1.0
    )

Writing Support-Assistant/support_assistant/schemas.py


In [96]:
import os

os.makedirs('Support-Assistant/support_assistant/docs', exist_ok=True)

with open('Support-Assistant/support_assistant/docs/doc_01.txt', 'w') as f:
    f.write('Delivery Policy: "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer\'s delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes."')

In [97]:
%%writefile Support-Assistant/support_assistant/docs/doc_02.txt

Returns & Refunds: "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto."

Writing Support-Assistant/support_assistant/docs/doc_02.txt


In [98]:
%%writefile Support-Assistant/support_assistant/docs/doc_03.txt

 Membership Tiers: "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period."

Writing Support-Assistant/support_assistant/docs/doc_03.txt


In [99]:
%%writefile Support-Assistant/support_assistant/docs/doc_04.txt

Order Tracking: "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue."

Writing Support-Assistant/support_assistant/docs/doc_04.txt


In [100]:
%%writefile Support-Assistant/support_assistant/docs/doc_05.txt

Order Cancellation Policy: "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee."

Writing Support-Assistant/support_assistant/docs/doc_05.txt


In [101]:
%%writefile Support-Assistant/support_assistant/docs/doc_06.txt

 Damaged or Missing Items: "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed."

Writing Support-Assistant/support_assistant/docs/doc_06.txt


In [102]:
%%writefile Support-Assistant/support_assistant/docs/doc_07.txt

 Gift Cards: "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law."

Writing Support-Assistant/support_assistant/docs/doc_07.txt


In [103]:
%%writefile Support-Assistant/support_assistant/docs/doc_08.txt

 Customer Support Hours: "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."

Writing Support-Assistant/support_assistant/docs/doc_08.txt


In [104]:
%%writefile Support-Assistant/support_assistant/embeddings.py

from functools import lru_cache

from sentence_transformers import SentenceTransformer

from .config import EMBEDDING_MODEL_NAME


@lru_cache(maxsize=1)
def get_embedding_model() -> SentenceTransformer:
    """
    Load the embedding model once and reuse it.

    The model runs locally.
    No LLM API is used here.
    """

    print(
        f"Loading embedding model: {EMBEDDING_MODEL_NAME}"
    )

    model = SentenceTransformer(
        EMBEDDING_MODEL_NAME
    )

    return model


def embed_text(text: str) -> list[float]:
    """
    Convert text into an embedding vector.
    """

    model = get_embedding_model()

    embedding = model.encode(
        text,
        normalize_embeddings=True
    )

    return embedding.tolist()


def embed_documents(
    documents: list[str]
) -> list[list[float]]:
    """
    Convert multiple documents into embedding vectors.
    """

    model = get_embedding_model()

    embeddings = model.encode(
        documents,
        normalize_embeddings=True
    )

    return embeddings.tolist()

Writing Support-Assistant/support_assistant/embeddings.py


In [105]:
%%writefile Support-Assistant/support_assistant/ingest.py

from pathlib import Path

import chromadb

from .config import (
    CHROMA_DIR,
    COLLECTION_NAME,
    DOCS_DIR
)

from .embeddings import embed_documents


def load_documents() -> tuple[list[str], list[str], list[dict]]:
    """
    Load all corpus documents.

    Since the provided documents are short, we use one
    document = one chunk.

    Returns:
        ids
        documents
        metadatas
    """

    files = sorted(
        DOCS_DIR.glob("doc_*.txt")
    )

    if len(files) != 8:
        raise RuntimeError(
            f"Expected exactly 8 documents, "
            f"but found {len(files)} in {DOCS_DIR}"
        )

    ids = []
    documents = []
    metadatas = []

    for file_path in files:

        text = file_path.read_text(
            encoding="utf-8"
        ).strip()

        if not text:
            raise ValueError(
                f"Document is empty: {file_path}"
            )

        doc_id = file_path.stem

        # One document = one chunk.
        chunk_id = f"{doc_id}_chunk_01"

        ids.append(chunk_id)
        documents.append(text)

        metadatas.append(
            {
                "doc_id": doc_id,
                "source_file": file_path.name,
                "chunk_id": chunk_id
            }
        )

    return ids, documents, metadatas


def create_chroma_collection():
    """
    Create a persistent local ChromaDB collection.
    """

    CHROMA_DIR.mkdir(
        parents=True,
        exist_ok=True
    )

    client = chromadb.PersistentClient(
        path=str(CHROMA_DIR)
    )

    # Delete old collection so ingestion is deterministic.
    try:
        client.delete_collection(
            name=COLLECTION_NAME
        )

        print(
            f"Deleted existing collection: "
            f"{COLLECTION_NAME}"
        )

    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={
            "description": (
                "Zepto support policy corpus "
                "for Module 3"
            )
        }
    )

    return collection


def ingest():
    """
    Complete ingestion pipeline.
    """

    print("=" * 60)
    print("ZEpto Support Assistant - Ingestion")
    print("=" * 60)

    ids, documents, metadatas = load_documents()

    print(
        f"Loaded {len(documents)} documents."
    )

    collection = create_chroma_collection()

    print("Creating embeddings...")

    embeddings = embed_documents(
        documents
    )

    collection.add(
        ids=ids,
        documents=documents,
        metadatas=metadatas,
        embeddings=embeddings
    )

    print(
        f"Inserted {len(ids)} chunks into ChromaDB."
    )

    print(
        f"Collection: {COLLECTION_NAME}"
    )

    print(
        f"Database path: {CHROMA_DIR}"
    )

    print("=" * 60)
    print("INGESTION COMPLETED SUCCESSFULLY")
    print("=" * 60)


if __name__ == "__main__":
    ingest()

Writing Support-Assistant/support_assistant/ingest.py


In [106]:
%%writefile Support-Assistant/support_assistant/retriever.py

import chromadb

from .config import (
    CHROMA_DIR,
    COLLECTION_NAME,
    TOP_K
)

from .embeddings import embed_text


def get_chroma_collection():
    """
    Connect to the persistent ChromaDB collection.
    """

    client = chromadb.PersistentClient(
        path=str(CHROMA_DIR)
    )

    try:
        collection = client.get_collection(
            name=COLLECTION_NAME
        )
    except Exception as exc:
        raise RuntimeError(
            "ChromaDB collection does not exist. "
            "Run ingestion first:\n\n"
            "python -m support_assistant.ingest"
        ) from exc

    return collection


def retrieve(
    query: str,
    top_k: int = TOP_K
) -> tuple[str, list[str]]:
    """
    Retrieve the most similar chunk from ChromaDB.

    Returns:
        retrieved_context
        sources
    """

    collection = get_chroma_collection()

    query_embedding = embed_text(
        query
    )

    result = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=[
            "documents",
            "metadatas",
            "distances"
        ]
    )

    documents = result.get(
        "documents",
        [[]]
    )[0]

    ids = result.get(
        "ids",
        [[]]
    )[0]

    if not documents:
        return "", []

    retrieved_context = "\n\n".join(
        documents
    )

    return retrieved_context, ids

Writing Support-Assistant/support_assistant/retriever.py


In [107]:
%%writefile Support-Assistant/support_assistant/prompts.py

def build_support_prompt(
    query: str,
    context: str,
    task: str = "Answer the customer's Zepto policy question.",
    length: str = "Keep the answer concise and helpful."
) -> str:
    """
    Structured prompt following:

    role -> context -> task -> format -> length

    Includes:
    - explicit negative constraint
    - few-shot example
    """

    prompt = f"""
ROLE:
You are a helpful Zepto customer-support assistant.

CONTEXT:
You may answer policy questions only from the provided
retrieved context.

TASK:
{task}

IMPORTANT CONSTRAINT:
Do not answer using information that is not present
in the provided context. Do not invent Zepto policies,
prices, timings, refunds, membership benefits, or
support procedures.

FORMAT:
Return a direct customer-friendly answer in plain text.

LENGTH:
{length}

FEW-SHOT EXAMPLE:

Example user question:
"How long does a refund take?"

Example context:
"Approved refunds are credited to the original payment
method within 3–5 business days."

Example answer:
"Approved refunds are credited to the original payment
method within 3–5 business days."

END FEW-SHOT EXAMPLE.

CURRENT USER QUERY:
{query}

RETRIEVED CONTEXT:
{context}
"""

    return prompt.strip()

Writing Support-Assistant/support_assistant/prompts.py


In [108]:
%%writefile Support-Assistant/support_assistant/llm.py
from .config import (
    GROQ_API_KEY,
    GROQ_MODEL,
    MOCK_LLM,
    MOCK_CONFIDENCE,
    SNIPPET_LENGTH
)

from .prompts import build_support_prompt


# ---------------------------------------------------------
# Intent keywords
# ---------------------------------------------------------

POLICY_KEYWORDS = {
    "delivery": [
        "delivery",
        "deliver",
        "delivered",
        "delivery fee",
        "priority delivery"
    ],

    "return": [
        "return",
        "returns",
        "returning"
    ],

    "refund": [
        "refund",
        "refunded",
        "money back"
    ],

    "membership": [
        "membership",
        "member",
        "pass",
        "pass+",
        "subscription"
    ],

    "tracking": [
        "tracking",
        "track order",
        "rider",
        "where is my order",
        "order status"
    ],

    "cancel": [
        "cancel",
        "cancellation",
        "cancelled",
        "canceled"
    ],

    "gift card": [
        "gift card",
        "giftcard",
        "gift voucher"
    ],

    "support hours": [
        "support hours",
        "customer support",
        "support",
        "phone support",
        "email support",
        "chat support"
    ]
}


def classify_with_mock(query: str) -> str:
    """
    Deterministic keyword heuristic required by
    the graded MOCK_LLM=1 baseline.
    """

    query_lower = query.lower()

    for category, keywords in POLICY_KEYWORDS.items():

        for keyword in keywords:

            if keyword in query_lower:
                return "policy_question"

    return "general_question"


def classify_with_llm(query: str) -> str:
    """
    Optional real-LLM classification.

    This function is NOT used when MOCK_LLM=1.
    """

    if not GROQ_API_KEY:
        raise RuntimeError(
            "MOCK_LLM=0 but GROQ_API_KEY is not set."
        )

    try:
        from groq import Groq

        client = Groq(
            api_key=GROQ_API_KEY
        )

        system_prompt = """
You classify Zepto customer support questions.

Return exactly one of:

policy_question
general_question

Use policy_question when the question concerns:
delivery, returns, refunds, membership, tracking,
cancellation, gift cards, or support hours.

Otherwise return:
general_question
"""

        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "system",
                    "content": system_prompt
                },
                {
                    "role": "user",
                    "content": query
                }
            ],
            temperature=0
        )

        result = response.choices[0].message.content.strip()

        if "policy_question" in result:
            return "policy_question"

        return "general_question"

    except Exception:
        # Safe fallback.
        return classify_with_mock(query)


def classify_intent(query: str) -> str:
    """
    Select mock or real classification according to
    MOCK_LLM.
    """

    if MOCK_LLM:
        return classify_with_mock(query)

    return classify_with_llm(query)


def mock_retrieval_answer(
    query: str,
    context: str
) -> tuple[str, float]:
    """
    Deterministic mock generation.

    Required graded output:
    "Based on the retrieved context: {top_chunk_snippet}"
    """

    if not context:
        return (
            "Based on the retrieved context: "
            "No relevant policy context was retrieved.",
            MOCK_CONFIDENCE
        )

    snippet = context[
        :SNIPPET_LENGTH
    ].replace("\n", " ").strip()

    answer = (
        "Based on the retrieved context: "
        f"{snippet}"
    )

    return answer, MOCK_CONFIDENCE


def mock_direct_answer() -> tuple[str, float]:
    """
    Deterministic response for general questions.
    """

    answer = (
        "I can only answer questions about Zepto "
        "policies right now."
    )

    return answer, MOCK_CONFIDENCE


def real_retrieval_answer(
    query: str,
    context: str
) -> tuple[str, float]:
    """
    Optional real LLM generation for retrieved questions.
    """

    if not GROQ_API_KEY:
        raise RuntimeError(
            "MOCK_LLM=0 but GROQ_API_KEY is not set."
        )

    prompt = build_support_prompt(
        query=query,
        context=context,
        task=(
            "Answer the customer's question using "
            "only the retrieved Zepto policy context."
        ),
        length=(
            "Use 1 to 3 short sentences."
        )
    )

    try:
        from groq import Groq

        client = Groq(
            api_key=GROQ_API_KEY
        )

        response = client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            temperature=0
        )

        answer = (
            response
            .choices[0]
            .message
            .content
            .strip()
        )

        return answer, 1.0

    except Exception as exc:
        raise RuntimeError(
            f"Real LLM generation failed: {exc}"
        ) from exc


def generate_retrieval_answer(
    query: str,
    context: str
) -> tuple[str, float]:
    """
    Generate answer for policy questions.
    """

    if MOCK_LLM:
        return mock_retrieval_answer(
            query,
            context
        )

    return real_retrieval_answer(
        query,
        context
    )


def generate_direct_answer() -> tuple[str, float]:
    """
    Generate answer for general questions.
    """

    if MOCK_LLM:
        return mock_direct_answer()

    # Optional real-LLM behavior.
    return mock_direct_answer()

Writing Support-Assistant/support_assistant/llm.py


In [109]:
%%writefile Support-Assistant/support_assistant/graph.py

from typing import Literal

from langgraph.graph import (
    StateGraph,
    START,
    END
)

from .state import SupportState

from .llm import (
    classify_intent as llm_classify_intent,
    generate_retrieval_answer,
    generate_direct_answer
)

from .retriever import retrieve


# Node 1: classify_intent

def classify_intent_node(
    state: SupportState
) -> SupportState:
    """
    Classify incoming query as:

    policy_question
    or
    general_question
    """

    query = state["query"]

    intent = llm_classify_intent(
        query
    )

    if intent == "policy_question":

        route = "retrieve_and_answer"

    else:

        route = "direct_answer"

    return {
        **state,
        "intent": intent,
        "route": route
    }


# Node 2: retrieve_and_answer

def retrieve_and_answer_node(
    state: SupportState
) -> SupportState:
    """
    Retrieve the most relevant ChromaDB chunk and
    generate an answer.

    MOCK_LLM branching occurs inside the generation
    function.
    """

    query = state["query"]

    context, sources = retrieve(
        query
    )

    answer, confidence = (
        generate_retrieval_answer(
            query=query,
            context=context
        )
    )

    return {
        **state,
        "retrieved_context": context,
        "sources": sources,
        "answer": answer,
        "confidence": confidence
    }


# Node 3: direct_answer

def direct_answer_node(
    state: SupportState
) -> SupportState:
    """
    Handle unrelated/general questions without retrieval.
    """

    answer, confidence = (
        generate_direct_answer()
    )

    return {
        **state,
        "sources": [],
        "retrieved_context": "",
        "answer": answer,
        "confidence": confidence
    }


# Conditional routing

def route_after_classification(
    state: SupportState
) -> Literal[
    "retrieve_and_answer",
    "direct_answer"
]:

    if state["intent"] == "policy_question":
        return "retrieve_and_answer"

    return "direct_answer"


# Build LangGraph

def build_graph():
    """
    Build and compile the LangGraph StateGraph.
    """

    graph = StateGraph(
        SupportState
    )

    # Required 3 nodes
    graph.add_node(
        "classify_intent",
        classify_intent_node
    )

    graph.add_node(
        "retrieve_and_answer",
        retrieve_and_answer_node
    )

    graph.add_node(
        "direct_answer",
        direct_answer_node
    )

    # Start -> classify
    graph.add_edge(
        START,
        "classify_intent"
    )

    # Conditional edge
    graph.add_conditional_edges(
        "classify_intent",
        route_after_classification,
        {
            "retrieve_and_answer":
                "retrieve_and_answer",

            "direct_answer":
                "direct_answer"
        }
    )

    # Both branches finish
    graph.add_edge(
        "retrieve_and_answer",
        END
    )

    graph.add_edge(
        "direct_answer",
        END
    )

    return graph.compile()


# Create reusable compiled graph
support_graph = build_graph()

Writing Support-Assistant/support_assistant/graph.py


In [110]:
%%writefile Support-Assistant/support_assistant/main.py

from fastapi import (
    FastAPI,
    HTTPException
)

from .config import MOCK_LLM

from .schemas import (
    AskRequest,
    AskResponse
)

from .graph import support_graph


app = FastAPI(
    title="Zepto Support Assistant",
    description=(
        "Module 3 GenAI Support Assistant using "
        "LangGraph, ChromaDB, Sentence Transformers "
        "and FastAPI."
    ),
    version="1.0.0"
)


@app.get("/")
def root():
    """
    Basic health endpoint.
    """

    return {
        "service": "Zepto Support Assistant",
        "status": "running",
        "mock_llm": MOCK_LLM
    }


@app.get("/health")
def health():
    """
    Health check.
    """

    return {
        "status": "ok",
        "mock_llm": MOCK_LLM
    }


@app.post(
    "/ask",
    response_model=AskResponse
)
def ask(
    request: AskRequest
):
    """
    Main support assistant endpoint.

    Input:
        {"query": "How much is delivery?"}

    Output:
        {
            "answer": "...",
            "sources": [...],
            "confidence": 1.0
        }
    """

    query = request.query.strip()

    if not query:
        raise HTTPException(
            status_code=400,
            detail="Query cannot be empty."
        )

    try:

        result = support_graph.invoke(
            {
                "query": query
            }
        )

        response = AskResponse(
            answer=result["answer"],
            sources=result.get(
                "sources",
                []
            ),
            confidence=result.get(
                "confidence",
                1.0
            )
        )

        return response

    except Exception as exc:

        raise HTTPException(
            status_code=500,
            detail=str(exc)
        ) from exc

Writing Support-Assistant/support_assistant/main.py


In [111]:
import os
from pathlib import Path

requirements_content = """fastapi>=0.115.0
uvicorn[standard]>=0.30.0
pydantic>=2.7.0

langgraph>=0.2.0

chromadb>=0.5.0

sentence-transformers>=3.0.0

torch>=2.0.0

groq>=0.9.0

opentelemetry-api==1.42.1
opentelemetry-sdk==1.42.1
protobuf==5.27.2
"""

file_path = Path('Support-Assistant/support_assistant/requirements.txt')
os.makedirs(file_path.parent, exist_ok=True)
with open(file_path, 'w') as f:
    f.write(requirements_content)

In [112]:
%%writefile Support-Assistant/support_assistant/Dockerfile

FROM python:3.11-slim

# Environment

ENV PYTHONDONTWRITEBYTECODE=1
ENV PYTHONUNBUFFERED=1

# Required graded baseline
ENV MOCK_LLM=1

# Working directory

WORKDIR /app

# System dependencies

RUN apt-get update && \
    apt-get install -y --no-install-recommends \
    build-essential \
    && rm -rf /var/lib/apt/lists/*

# Copy project

COPY . /app

# Install Python dependencies

RUN pip install --no-cache-dir \
    -r requirements.txt

# Build ChromaDB during image creation

RUN python -m support_assistant.ingest

# API port

EXPOSE 7860

# Start FastAPI


CMD [
    "uvicorn",
    "support_assistant.main:app",
    "--host",
    "0.0.0.0",
    "--port",
    "7860"
]

Writing Support-Assistant/support_assistant/Dockerfile


In [113]:
%cd /content/Support-Assistant


/content/Support-Assistant


In [114]:
%%writefile .gitignore

__pycache__/
*.py[cod]

.venv/
venv/
env/

.ipynb_checkpoints/

.env

.DS_Store

*.log

.pytest_cache/

support_assistant/chroma_db/*
!support_assistant/chroma_db/.gitkeep

Overwriting .gitignore


In [115]:
import os
from pathlib import Path

gitkeep_file_path = Path('support_assistant/chroma_db/.gitkeep')


os.makedirs(gitkeep_file_path.parent, exist_ok=True)

with open(gitkeep_file_path, 'w') as f:
    f.write('')

In [116]:
%cd /content/Support-Assistant

/content/Support-Assistant


In [117]:
%%writefile README.md
# Zepto Support Assistant — Module 3

## Overview

This project implements a small GenAI support assistant for Zepto.

The system uses:

- Sentence Transformers
- all-MiniLM-L6-v2
- ChromaDB
- LangGraph
- TypedDict
- FastAPI
- Pydantic
- Deterministic MOCK_LLM baseline

The default configuration is:

```text
MOCK_LLM=1

Overwriting README.md


In [118]:
!pip install --force-reinstall -r support_assistant/requirements.txt

  Using cached fastapi-0.141.1-py3-none-any.whl.metadata (27 kB)
  Using cached uvicorn-0.52.3-py3-none-any.whl.metadata (6.6 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langgraph-1.2.11-py3-none-any.whl.metadata (4.9 kB)
  Using cached chromadb-1.5.9-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.0 kB)
  Using cached sentence_transformers-5.7.0-py3-none-any.whl.metadata (18 kB)
  Using cached torch-2.13.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (38 kB)
  Using cached groq-1.6.0-py3-none-any.whl.metadata (16 kB)
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached typing_extensions-4.16.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
  Using cached typing_inspect

In [119]:
!pip show opentelemetry-api
!pip show opentelemetry-sdk
!pip show protobuf

ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 46, in run
    if not print_results(
           ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 178, in print_results
    for i, dist in enumerate(distributions):
                   ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 114, in search_packages_info
    required_by = sorted(_get_requiring_packages(dist), key=str.lower)
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/show.py", line 95, in <genexpr>
    in {canonicalize_name(d.name) for d in dist.i

In [120]:
!python -m support_assistant.ingest

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/content/Support-Assistant/support_assistant/ingest.py", line 12, in <module>
    from .embeddings import embed_documents
  File "/content/Support-Assistant/support_assistant/embeddings.py", line 4, in <module>
    from sentence_transformers import SentenceTransformer
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/__init__.py", line 15, in <module>
    from sentence_transformers.base.sampler import DefaultBatchSampler, MultiDatasetDefaultBatchSampler
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/base/__init__.py", line 4, in <module>
    from .model import BaseModel
  File "/usr/local/lib/python3.12/dist-packages/sentence_transformers/base/model.py", line 24, in <module>
  File "<frozen importlib._bootstrap>", line 1412, in _handle_fromlist
  File "/usr/local/lib/python3.12/dist-packages/transf

In [121]:
!pip show sentence-transformers

Name: sentence-transformers
Version: 5.6.0
Summary: Embeddings, Retrieval, and Reranking
Home-page: https://www.SBERT.net
Author: 
Author-email: Nils Reimers <info@nils-reimers.de>, Tom Aarsen <tom.aarsen@huggingface.co>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: huggingface-hub, numpy, scikit-learn, scipy, torch, tqdm, transformers, typing_extensions
Required-by: 


In [122]:
print("all-MiniLM-L6-v2")

all-MiniLM-L6-v2


In [123]:
TOP_K = 1

In [124]:
prompt_component_labels = [
    "role",
    "context",
    "task",
    "format",
    "length"
]

In [125]:
important_constraint = """Do not answer using information that is not present
in the provided context."""

In [126]:
graph_nodes = [
    "classify_intent",
    "retrieve_and_answer",
    "direct_answer"
]

In [127]:
MOCK_LLM=1
classification_examples = [
    ("How much is the delivery fee?", "policy_question"),
    ("What is the capital of India?", "general_question")
]

In [128]:
MOCK_LLM = 1
mock_retrieval_template = "Based on the retrieved context: {top_chunk_snippet}"
mock_direct_answer_text = "I can only answer questions about Zepto policies right now."
mock_confidence_value = 1.0

In [129]:
!nohup uvicorn support_assistant.main:app --host 0.0.0.0 --port 7860 > uvicorn.log 2>&1 &

In [130]:
with open('uvicorn.log', 'r') as f:
    print(f.read())

In [131]:
!curl -v -X POST "http://127.0.0.1:7860/ask" \
-H "Content-Type: application/json" \
-d "{\"query\":\"How long does a refund take?\"}"

Note: Unnecessary use of -X or --request, POST is already inferred.
*   Trying 127.0.0.1:7860...
* connect to 127.0.0.1 port 7860 failed: Connection refused
* Failed to connect to 127.0.0.1 port 7860 after 0 ms: Connection refused
* Closing connection 0
curl: (7) Failed to connect to 127.0.0.1 port 7860 after 0 ms: Connection refused


In [132]:
!curl -X POST "http://127.0.0.1:7860/ask" \
-H "Content-Type: application/json" \
-d "{\"query\":\"What is the capital of India?\"}"

curl: (7) Failed to connect to 127.0.0.1 port 7860 after 0 ms: Connection refused


In [133]:
!docker build -t zepto-support .

/bin/bash: line 1: docker: command not found


In [134]:
!docker run --rm -p 7860:7860 zepto-support

/bin/bash: line 1: docker: command not found


In [135]:
# http://localhost:7860
#http://localhost:7860/docs

In [137]:
import requests

api_url = "http://127.0.0.1:7860/ask"


payload = {
    "query": "How much is delivery?"
}

headers = {
    "Content-Type": "application/json"
}

response = requests.post(api_url, json=payload, headers=headers)

print(response.json())

ConnectionError: HTTPConnectionPool(host='127.0.0.1', port=7860): Max retries exceeded with url: /ask (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=7860): Failed to establish a new connection: [Errno 111] Connection refused"))